In [ ]:
import pandas as pd
import numpy as np
import time
import multiprocessing
from collections import Counter
from joblib import Parallel, delayed
import dask.bag as db
from dask.distributed import Client

In [ ]:
def prepare_data(file_path):
    df = pd.read_csv(file_path)
    
    # Define columns that are not features (identifiers or targets) and should be ignored
    ignored_cols = ['Student_ID', 'Final_Grade', 'Letter_Grade', 'AtRisk']
    
    # Create the feature matrix by dropping the ignored columns
    X_raw = df.drop(columns=ignored_cols, errors='ignore')
    
    # Extract the target variable (labels) indicating if a student is At Risk
    y = df['AtRisk'].values
    
    # Convert categorical variables into dummy/indicator variables (One-Hot Encoding)
    X_numeric = pd.get_dummies(X_raw, drop_first=True).astype(float).values
    
    return X_numeric, y

def normalize_features(X):
    x_min = X.min(axis=0)
    x_max = X.max(axis=0)
    
    return (X - x_min) / (x_max - x_min + 1e-10)

# Intentionally heavy brute-force KNN.
# Processes each query one-by-one with full sorting.
def knn_predict_serial(X_train, y_train, X_test, k=5):
    predictions = []
    
    # Iterate through every single query point in the test set one by one
    for i in range(len(X_test)):
        query = X_test[i]
        
        # Distance Calculation
        distances = np.sqrt(np.sum((X_train - query)**2, axis=1))
        
        # Full sorting (The Bottleneck)
        nearest_indices = np.argsort(distances)[:k]
        
        # Majority voting
        # Retrieve the labels of the k nearest neighbors using the indices found above
        neighbor_labels = y_train[nearest_indices]
        # Use Counter to find the most frequent label among the neighbors
        most_common = Counter(neighbor_labels).most_common(1)
        predictions.append(most_common[0][0])
        
        # Print progress every 100 samples to monitor execution
        if (i + 1) % 100 == 0:
            print(f"Progress: {i + 1}/{len(X_test)} classified...")

    return predictions

if __name__ == "__main__":
    # Load and prepare the data
    X, y = prepare_data('data/realistic_student_data.csv')
    # Scale features to ensure distance calculations are not biased by feature magnitude
    X_scaled = normalize_features(X)
    
    # Define the number of samples to use for testing
    num_test = 500
    # Split the data: everything before the last 500 is training, the last 500 is testing
    train_end = len(X_scaled) - num_test
    X_train, y_train = X_scaled[:train_end], y[:train_end]
    X_test, y_test = X_scaled[train_end:], y[train_end:]

    print(f"Starting sequential KNN on {len(X_train)} training samples...")

    start_time = time.time()
    y_pred = knn_predict_serial(X_train, y_train, X_test, k=5)
    duration = time.time() - start_time

    # Evaluate
    accuracy = np.mean(y_pred == y_test)
    print("-" * 40)
    print(f"Total Serial Time: {duration:.4f} seconds")
    print(f"Final Accuracy: {accuracy * 100:.2f}%")

Starting sequential KNN on 19500 training samples...
Progress: 100/500 classified...
Progress: 200/500 classified...
Progress: 300/500 classified...
Progress: 400/500 classified...
Progress: 500/500 classified...
----------------------------------------
Total Serial Time: 1.0375 seconds
Final Accuracy: 90.40%


In [ ]:
if __name__ == "__main__":
    # Baseline recorded from your Sequential run
    BASELINE_TIME = 1.0375
    
    X_all, y_all = prepare_data('data/realistic_student_data.csv')
    X_scaled = normalize_features(X_all)
    X_train, y_train = X_scaled[:19500], y_all[:19500]
    X_test, y_test = X_scaled[19500:], y_all[19500:]

    num_cores = 12 
    k_value = 5

    print(f"Distributing 500 test instances into {num_cores} subsets...")
    
    test_chunks = np.array_split(X_test, num_cores)

    start_time = time.time()

    # Create a Pool of 12 workers to bypass the Global Interpreter Lock (GIL)
    with multiprocessing.Pool(processes=num_cores) as pool:
        # Prepare arguments for each chunk: (chunk_data, X_train, y_train, k)
        tasks = [(X_train, y_train, chunk, k_value) for chunk in test_chunks]
        
        # starmap distributes the chunks across the workers
        results_list = pool.starmap(knn_predict_serial, tasks)

    parallel_duration = time.time() - start_time
    
    # Flatten results
    y_pred = [item for sublist in results_list for item in sublist]
    

    accuracy = np.mean(np.array(y_pred) == y_test)
    print("-" * 45)
    print(f"Parallel Execution Time (12 Cores): {parallel_duration:.4f} seconds")
    # Speedup is calculated relative to your provided baseline
    print(f"Speedup vs Sequential: {BASELINE_TIME / parallel_duration:.2f}x")
    print(f"Final Accuracy: {accuracy * 100:.2f}%")


Distributing 500 test instances into 12 subsets...
---------------------------------------------
Parallel Execution Time (12 Cores): 0.8785 seconds
Speedup vs Sequential: 1.18x
Final Accuracy: 90.40%


In [5]:

if __name__ == "__main__":
    num_cores = 12
    k_value = 5
    
    X_all, y_all = prepare_data('data/realistic_student_data.csv')
    X_scaled = normalize_features(X_all)
    X_train, y_train = X_scaled[:19500], y_all[:19500]
    X_test, y_test = X_scaled[19500:], y_all[19500:]

    # Chunk the test data
    test_chunks = np.array_split(X_test, num_cores)

    print(f"Executing Joblib Scaling on {num_cores} cores...")
    
    start_time = time.time()

    # Joblib implementation
    # Parallel(n_jobs=) manages the pool
    # delayed(function)(args) captures the task
    results_list = Parallel(n_jobs=num_cores)(
        delayed(knn_predict_serial)(X_train, y_train, chunk, k_value) 
        for chunk in test_chunks
    )

    duration = time.time() - start_time
    
    # Flatten results
    y_pred = [item for sublist in results_list for item in sublist]
    
    accuracy = np.mean(np.array(y_pred) == y_test)
    print("-" * 45)
    print(f"Joblib Execution Time: {duration:.4f} seconds")
    print(f"Final Accuracy: {accuracy * 100:.2f}%")

Executing Joblib Scaling on 12 cores...
---------------------------------------------
Joblib Execution Time: 1.1923 seconds
Final Accuracy: 90.40%


In [8]:
# Update the function signature so the 'chunk' comes first
def knn_predict_serial_updated(test_subset, X_train, y_train, k=5):
    predictions = []
    for query in test_subset:
        # Distance calculation
        distances = np.sqrt(np.sum((X_train - query)**2, axis=1))
        # Heavy Sort
        nearest_indices = np.argsort(distances)[:k]
        # Voting
        neighbor_labels = y_train[nearest_indices]
        most_common = Counter(neighbor_labels).most_common(1)
        predictions.append(most_common[0][0])
    return predictions

if __name__ == "__main__":
    # Start a Dask Local Cluster
    # This allows you to view the Dask Dashboard (usually at http://localhost:8787)
    client = Client(n_workers=12, threads_per_worker=1)
    print(f"Dask Dashboard available at: {client.dashboard_link}")

    X_all, y_all = prepare_data('data/realistic_student_data.csv')
    X_scaled = normalize_features(X_all)
    X_train, y_train = X_scaled[:19500], y_all[:19500]
    X_test, y_test = X_scaled[19500:], y_all[19500:]

    # Divide test data into chunks
    num_chunks = 12
    test_chunks = np.array_split(X_test, num_chunks)

    # Create a Dask Bag (The Dask Data Structure)
    # This distributes the chunks across the Dask cluster
    b = db.from_sequence(test_chunks, npartitions=num_chunks)

    print(f"Starting Dask KNN Execution...")

    start_time = time.time()

    # Map the worker function across the bag
    delayed_predictions = b.map(knn_predict_serial_updated, X_train=X_train, y_train=y_train, k=5)

    # Execute the Dask graph
    results_list = delayed_predictions.compute()
    duration = time.time() - start_time

    # Flatten results
    y_pred = [item for sublist in results_list for item in sublist]
    
    accuracy = np.mean(np.array(y_pred) == y_test)
    print("-" * 45)
    print(f"Dask Execution Time: {duration:.4f} seconds")
    print(f"Final Accuracy: {accuracy * 100:.2f}%")

Dask Dashboard available at: http://127.0.0.1:8787/status
Starting Dask KNN Execution...


---------------------------------------------
Dask Execution Time: 1.2660 seconds
Final Accuracy: 90.40%


In [9]:
client.close()